In [1]:
import jax
import subprocess

def get_gpu_memory_usage_mb():
    try:
        devices = jax.devices()
        if not devices or devices[0].platform != "gpu":
            return 0

        logical_id = devices[0].id  

        cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", None)
        if cuda_visible:
            physical_ids = [int(x) for x in cuda_visible.split(",")]
            physical_id = physical_ids[logical_id]
        else:
            physical_id = logical_id

        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        mem_values = [int(x) for x in result.strip().split('\n')]

        return mem_values[physical_id]
    except Exception as e:
        print("[get_gpu_memory_usage_mb] get_gpu_memory failed：", e)
        return 0


In [2]:
import os
# 选择GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.chdir('/home/zheling/Character_Recognition')

print("当前工作目录：", os.getcwd())


当前工作目录： /home/zheling/Character_Recognition


In [3]:
from typing import Dict, List, Tuple, Optional
from functools import partial
import os
import struct
from tqdm import tqdm
import jax 
import jax.numpy as jnp
import jax.nn as jnn
import jax.random as jr
import jax.lax as lax
import numpy as np
import optax 
import equinox as eqx
import signax
from signax import signature

import math
import matplotlib.pyplot as plt
import random

from math import sqrt, atan2
from collections import defaultdict

import time
import subprocess
import json
import traceback
import gc
import itertools
import subprocess
import pandas as pd
from scipy import stats

In [4]:
import jax
import subprocess

def get_gpu_memory_usage_mb():
    try:
        devices = jax.devices()
        if not devices or devices[0].platform != "gpu":
            return 0
        gpu_id = devices[0].id 
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
            encoding='utf-8'
        )
        mem_values = [int(x) for x in result.strip().split('\n')]
        return mem_values[gpu_id]
    except Exception:
        return 0


In [5]:
def _concat_points(strokes):
    if len(strokes) == 0:
        return np.zeros((0, 2), dtype=np.float32), []
    idx = []
    all_pts = []
    start = 0
    for s in strokes:
        s = np.asarray(s, dtype=np.float32)
        L = len(s)
        if L == 0:
            continue
        all_pts.append(s)
        idx.append((start, start + L))
        start += L

    if not all_pts:
        return np.zeros((0, 2), dtype=np.float32), []

    return np.concatenate(all_pts, axis=0), idx

def _rotate_pts_about_anchor(P, theta, anchor):
    c, s = np.cos(theta), np.sin(theta)
    xy = P[:, :2]
    anc_xy = anchor[:2]

    D = xy - anc_xy
    x_new = c * D[:, 0] - s * D[:, 1]
    y_new = s * D[:, 0] + c * D[:, 1]

    R = np.stack([x_new, y_new], axis=1) + anc_xy

    if P.shape[1] > 2:
        return np.concatenate([R, P[:, 2:]], axis=1)
    return R

def apply_hanging_on_strokes(strokes, mode="SC"):
    """
    Hanging Normalization
    strokes: List[List[x, y, ...]]
    """
    if (mode is None) or (mode == "None") or (len(strokes) == 0):
        return strokes

    P, spans = _concat_points(strokes)
    if P.shape[0] < 2:
        return strokes

    first_pt = P[0]

    # SC 
    if mode == "SC":
        center = P[:, :2].mean(axis=0)
        v = center - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]

    # SE 
    elif mode == "SE":
        last_pt = P[-1]
        v = last_pt[:2] - first_pt[:2]
        angle = np.arctan2(v[1], v[0]) + np.pi / 2.0

        P2 = _rotate_pts_about_anchor(P, -angle, first_pt)
        return [P2[s:e] for (s, e) in spans]

    # ASE 
    elif mode == "ASE":
        out = []
        for s in strokes:
            s_arr = np.asarray(s, dtype=np.float32)
            if len(s_arr) < 2:
                out.append(s_arr)
                continue

            start = s_arr[0]
            end = s_arr[-1]
            v = end[:2] - start[:2]

            angle = np.arctan2(v[1], v[0]) + np.pi / 2.0
            s_rot = _rotate_pts_about_anchor(s_arr, -angle, start)
            out.append(s_rot)
        return out
    else:
        return strokes
        

In [6]:
def read_pot_file(filename):
    if not os.path.exists(filename):
        return [], ""
    raw_strokes = []
    with open(filename, "rb") as f:
        while True:
            sample_size = f.read(2)
            if sample_size == b'':
                break

            dword_code = f.read(2)
            if len(dword_code) < 2:
                break
            
            if dword_code[0] != 0:
                dword_code = bytes((dword_code[1], dword_code[0]))
            tag_code = struct.unpack(">H", dword_code)[0]

            skip = f.read(2)
            try:
                tag = struct.pack('>H', tag_code).decode("gbk")[0]
            except Exception:
                tag = str(tag_code)

            f.read(2)

            strokes_samples = []
            stroke_samples = []

            while True:
                bx = f.read(2)
                by = f.read(2)
                if bx == b'' or by == b'':
                    if stroke_samples:
                        strokes_samples.append(stroke_samples)
                        stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\xff\xff':
                    if stroke_samples:
                        strokes_samples.append(stroke_samples)
                        stroke_samples = []
                    break
                if bx == b'\xff\xff' and by == b'\x00\x00':
                    strokes_samples.append(stroke_samples)
                    stroke_samples = []
                    continue
                x = struct.unpack("<H", bx)[0]
                y = struct.unpack("<H", by)[0]
                stroke_samples.append((x, y))
            raw_strokes.append(strokes_samples)
    return raw_strokes, tag

def normalize_strokes(strokes, coord_range=1.0):
    if not strokes:
        return strokes
    all_points = [p for s in strokes for p in s]
    if not all_points:
        return strokes
    arr = np.array([np.asarray(p[:2], dtype=np.float32) for p in all_points])
    min_xy, max_xy = arr.min(0), arr.max(0)
    scale = max(max_xy[0] - min_xy[0], max_xy[1] - min_xy[1], 1e-6)

    out = []
    for s in strokes:
        new_s = []
        for p in s:
            extra = tuple(p[2:]) if len(p) > 2 else tuple()
            x = (p[0] - min_xy[0]) / scale * coord_range
            y = (p[1] - min_xy[1]) / scale * coord_range
            new_s.append((x, y) + extra)
        out.append(new_s)
    return out

def simplify_strokes(strokes, dist_thresh=5, angle_thresh=0.15):
    """RDP-like simplification."""
    out = []
    for s in strokes:
        if len(s) <= 2:
            out.append(s)
            continue
        simp = [s[0]]
        for i in range(1, len(s) - 1):
            prev, curr, next_p = simp[-1], s[i], s[i + 1]
            d1 = (curr[0] - prev[0], curr[1] - prev[1])
            d2 = (next_p[0] - curr[0], next_p[1] - curr[1])
            dist_sq = d1[0] ** 2 + d1[1] ** 2
            if d1 == (0, 0) or d2 == (0, 0):
                ang = 0.0
            else:
                ang = abs(math.atan2(d2[1], d2[0]) - math.atan2(d1[1], d1[0]))
            if dist_sq > dist_thresh ** 2 or ang > angle_thresh:
                simp.append(curr)
        simp.append(s[-1])
        out.append(simp)
    return out

def speed_normalization(strokes, target_speed=10.0):
    out = []
    for s in strokes:
        if len(s) <= 1:
            out.append(s)
            continue
        new_s = [s[0]]
        acc_dist = 0.0
        p1 = s[0]
        for i in range(1, len(s)):
            p2 = s[i]
            dist = math.hypot(p2[0] - p1[0], p2[1] - p1[1])
            acc_dist += dist
            curr_p1 = p1
            curr_dist = dist
            while acc_dist >= target_speed:
                acc_dist -= target_speed
                if curr_dist > 1e-6:
                    t = 1 - (acc_dist / curr_dist)
                    extra = tuple(curr_p1[2:]) if len(curr_p1) > 2 else tuple()
                    new_pt = (curr_p1[0] + t * (p2[0] - curr_p1[0]),
                              curr_p1[1] + t * (p2[1] - curr_p1[1])) + extra
                    new_s.append(new_pt)
                    curr_p1 = new_pt
                    # after interpolation, remaining distance becomes acc_dist which was carried
                    curr_dist = acc_dist
                else:
                    break
            p1 = p2
        new_s.append(s[-1])
        out.append(new_s)
    return out

def add_imaginary_stroke(strokes, add_ink_dim=True, add_pen_states=False):
    if len(strokes) < 2:
        return [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in strokes]

    total_len = 0.0
    count = 0
    for s in strokes:
        for i in range(len(s) - 1):
            total_len += math.hypot(s[i + 1][0] - s[i][0], s[i + 1][1] - s[i][1])
            count += 1
    avg_len = total_len / max(1, count)

    out = []
    for i in range(len(strokes)):
        out.append({"data": np.array(strokes[i], dtype=np.float32), "type": "real"})
        if i < len(strokes) - 1:
            start = strokes[i][-1]
            end = strokes[i + 1][0]
            dist = math.hypot(end[0] - start[0], end[1] - start[1])
            steps = max(1, int(math.ceil(dist / max(1e-6, avg_len))))
            v_stroke = []
            for j in range(steps + 1):
                alpha = j / steps
                extra = tuple(start[2:]) if len(start) > 2 else tuple()
                pt = (start[0] * (1 - alpha) + end[0] * alpha,
                      start[1] * (1 - alpha) + end[1] * alpha) + extra
                v_stroke.append(pt)
            out.append({"data": np.array(v_stroke, dtype=np.float32), "type": "virtual"})
    return out

def apply_data_augmentation(strokes, key):
    if not strokes:
        return strokes
    k_s, k_t, k_e = jr.split(key, 3)
    sx = 1.0 + float(jr.uniform(k_s, minval=-0.15, maxval=0.15))
    sy = 1.0 + float(jr.uniform(k_s, minval=-0.15, maxval=0.15))
    tx = float(jr.normal(k_t)) * 10.0
    ty = float(jr.normal(k_t)) * 10.0
    eps = float(jr.uniform(k_e, minval=0.0, maxval=2.0))

    all_pts = []
    for s in strokes:
        for p in s:
            all_pts.append(p[:2])

    if len(all_pts) == 0:
        return strokes
    arr = np.array(all_pts, dtype=np.float32)

    w, h = (arr.max(0) - arr.min(0)) + 1e-6
    fx, fy = 2 * math.pi / h, 2 * math.pi / w

    aug_strokes = []
    for s in strokes:
        new_stroke = []
        for p in s:
            x = p[0] * sx + tx + eps * math.sin(p[1] * sy * fx)
            y = p[1] * sy + ty + eps * math.sin(p[0] * sx * fy)
            new_pt = (x, y) + tuple(p[2:])
            new_stroke.append(new_pt)
        aug_strokes.append(new_stroke)
    return aug_strokes

def apply_rotation_on_strokes(strokes, angle_deg):
    """Rotates the entire character around the origin (0,0) by a specified angle."""
    if angle_deg is None or angle_deg == 0:
        return strokes
    theta = np.deg2rad(float(angle_deg))
    c, s = np.cos(theta), np.sin(theta)
    out = []
    for stroke in strokes:
        s_arr = np.array(stroke, dtype=np.float32)
        if len(s_arr) == 0:
            out.append(s_arr)
            continue
        x, y = s_arr[:, 0], s_arr[:, 1]
        xr = c * x - s * y
        yr = s * x + c * y
        if s_arr.shape[1] > 2:
            new_s = np.concatenate([np.stack([xr, yr], 1), s_arr[:, 2:]], 1)
        else:
            new_s = np.stack([xr, yr], 1)
        # convert back to list of tuples for consistency with other functions
        out.append([tuple(row) for row in np.asarray(new_s)])
    return out

def compute_derivatives_strokes(strokes):
    """
    Computes differential features for strokes based on differences.
    Output format per point: [x, y, (ink...), dx, dy, dxx, dyy]
    """
    seq = []
    for stroke in strokes:
        s_arr = np.array(stroke, dtype=np.float32)
        n = len(s_arr)
        if n == 0:
            continue
        xy = s_arr[:, :2]

        vd = np.zeros_like(xy)
        vc = np.zeros_like(xy)

        if n >= 2:
            vd[0] = xy[1] - xy[0]
            vd[-1] = xy[-1] - xy[-2]
            if n > 2:
                vd[1:-1] = xy[2:] - xy[:-2]

            vc[0] = vd[1] - vd[0] if n > 1 else 0
            vc[-1] = vd[-1] - vd[-2] if n > 1 else 0
            if n > 2:
                vc[1:-1] = vd[2:] - vd[:-2]
        feat = np.concatenate([s_arr, vd, vc], axis=1)
        for point in feat:
            seq.append(point)
    return np.array(seq, dtype=np.float32)

def build_feature_sequence(input_strokes, add_ink_dim, add_pen_states, use_derivatives):
    if len(input_strokes) > 0 and isinstance(input_strokes[0], dict):
        stroke_objs = input_strokes
    else:
        stroke_objs = [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in input_strokes]

    processed = []
    for item in stroke_objs:
        s, stype = item["data"], item["type"]
        xy = s[:, :2]

        if add_ink_dim:
            if stype == 'real':
                if s.shape[1] >= 3:
                    ink = s[:, 2:3]
                else:
                    ink = np.ones((len(s), 1), dtype=np.float32)
            else:
                ink = np.zeros((len(s), 1), dtype=np.float32)
            xy = np.concatenate([xy, ink], axis=1)

        if add_pen_states:
            pd = np.zeros((len(s), 1), dtype=np.float32)
            pu = np.zeros((len(s), 1), dtype=np.float32)
            if stype == 'real' and len(s) > 0:
                pd[0] = 1.0
                pu[-1] = 1.0
            xy = np.concatenate([xy, pd, pu], axis=1)
        processed.append(xy)

    if not processed:
        return np.zeros((0, 2), dtype=np.float32)

    if use_derivatives:
        final_seq = []
        for feat in processed:
            coords = feat[:, :2]
            n = len(coords)
            vd = np.zeros_like(coords)
            vc = np.zeros_like(coords)
            if n >= 2:
                vd[0] = coords[1] - coords[0]
                vd[-1] = coords[-1] - coords[-2]
                if n > 2:
                    vd[1:-1] = (coords[2:] - coords[:-2]) / 2.0
                vc[0] = vd[1] - vd[0]
                vc[-1] = vd[-1] - vd[-2]
                if n > 2:
                    vc[1:-1] = (vd[2:] - vd[:-2]) / 2.0
            final_seq.append(np.concatenate([feat, vd, vc], axis=1))
        return np.concatenate(final_seq, 0)

    return np.concatenate(processed, 0)

def normalize_all_dimensions(sequence, add_ink_dim, add_pen_states):
    if len(sequence) == 0:
        return sequence
    seq = sequence.copy()

    current_idx = 2
    if add_ink_dim:
        max_ink = seq[:, 2].max()
        if max_ink > 0:
            seq[:, 2] /= max_ink
        current_idx += 1

    if add_pen_states:
        current_idx += 2

    if current_idx < seq.shape[1]:
        derivs = seq[:, current_idx:]
        max_val = np.max(np.abs(derivs))
        if max_val > 1e-6:
            seq[:, current_idx:] /= max_val

    return seq

def process_sequence_length(seq, target_len, use_end_pad):
    L, D = seq.shape
    if L >= target_len:
        return seq[:target_len]
    out = np.zeros((target_len, D), dtype=np.float32)
    out[:L] = seq
    if use_end_pad and L > 0:
        out[L:] = seq[-1]
        if D >= 4:
            out[L:, -4:] = 0
    return out

@eqx.filter_jit
def compute_window_signatures(path, window_size=10, stride=1, depth=2):
    """
    Computes Path Signatures using sliding windows.
    """
    length, dim = path.shape
    if length < window_size:
        window_size = length
    num_windows = max(1, (length - window_size) // stride + 1)

    starts = jnp.arange(num_windows) * stride
    indices = starts[:, None] + jnp.arange(window_size)[None, :]
    indices = jnp.clip(indices, 0, length - 1)

    windows = path[indices]

    sigs = jax.vmap(lambda w: signax.signature(w, depth=depth))(windows)
    return sigs

def build_one_feature(strokes, cfg, key_aug=None, angle=None):
    if cfg.get('use_simplify', False):
        strokes = simplify_strokes(strokes, cfg.get('dist_thresh', 5), cfg.get('angle_thresh', 0.15))
    if cfg.get('use_speed_norm', False):
        strokes = speed_normalization(strokes, cfg.get('target_speed', 10.0))
    if angle:
        strokes = apply_rotation_on_strokes(strokes, angle)

    if key_aug is not None:
        strokes = apply_data_augmentation(strokes, key_aug)

    if cfg.get('use_hanging', False):
        strokes = apply_hanging_on_strokes(strokes, cfg.get('hanging_mode', "SC"))

    strokes = normalize_strokes(strokes, cfg.get('coord_range', 1.0))

    if cfg.get('add_imaginary_stroke', False) and len(strokes) > 1:
        strokes_data = add_imaginary_stroke(strokes, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False))
    else:
        strokes_data = [{"data": np.array(s, dtype=np.float32), "type": "real"} for s in strokes]

    seq = build_feature_sequence(strokes_data, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False), cfg.get('use_derivatives', False))
    seq = normalize_all_dimensions(seq, cfg.get('add_ink_dim', False), cfg.get('add_pen_states', False))

    seq = process_sequence_length(seq, cfg.get('target_length', 128), cfg.get('use_end_padding', True))

    if cfg.get('use_signatures', False):
        return compute_window_signatures(jnp.array(seq), cfg.get('window_size', 10), cfg.get('stride', 1), cfg.get('depth', 2))
    return seq

In [7]:
class ArrayDataloader:
    def __init__(self, X, y, a=None, batch_size=64):
        self.X = jnp.asarray(X)
        self.y = jnp.asarray(y)
        self.a = jnp.asarray(a) if a is not None else None
        self.size = len(X)
        self.batch_size = batch_size  
        self.datas = self.X
        self.labels = self.y
        self.angles = self.a if self.a is not None else jnp.zeros((self.size,))

    def loop(self, bs, key, shuffle=True):
        idx = jr.permutation(key, jnp.arange(self.size)) if shuffle else jnp.arange(self.size)
        for i in range(0, self.size, bs):
            b = idx[i:i + bs]
            yield self.X[b], self.y[b], (self.a[b] if self.a is not None else None)


def process_class(c_idx, data_dir, is_train, cfg, seed=None):
    raw, _ = read_pot_file(os.path.join(data_dir, f"{c_idx}.pot"))
    if not raw:
        return [], [], []  

    n_tr = int(0.8 * len(raw))
    sel = raw[:n_tr] if is_train else raw[n_tr:]

    if is_train:
        feats, labels, angles = [], [], []
        key = jr.PRNGKey(int(seed) if seed is not None else 42)
        for i, s in enumerate(sel):
            k1, k2 = jr.split(jr.fold_in(key, i))
            ang = float(jr.uniform(k1, minval=0, maxval=360))
            f = build_one_feature(s, cfg, key_aug=k2, angle=ang)
            feats.append(f)
            labels.append(c_idx - 1)
            angles.append(ang)
        return feats, labels, angles
    else:
        f30, l30, a30 = [], [], []
        ang_list = list(range(0, 360, 12))
        for i, s in enumerate(sel):
            for ang in ang_list:
                f30.append(build_one_feature(s, cfg, angle=float(ang)))
                l30.append(c_idx - 1)
                a30.append(float(ang))
        return f30, l30, a30



def make_loaders(data_dir, n_cls, cfg, seed=42, mode="train"):
    tr, te30 = ([], [], []), ([], [], [])

    for c in tqdm(range(1, n_cls + 1), desc=f"Build {mode} set"):
        if mode == "train":
            F, L, A = process_class(c, data_dir, True, cfg, seed)
            tr[0].extend(F)
            tr[1].extend(L)
            tr[2].extend(A)

        f30, l30, a30 = process_class(c, data_dir, False, cfg)
        te30[0].extend(f30)
        te30[1].extend(l30)
        te30[2].extend(a30)

    def _L(d):
        return ArrayDataloader(d[0], d[1], d[2])

    if mode == "train":
        tr_loader = _L(tr)
        te30_loader = _L(te30)
        dim = tr_loader.X.shape[2] if len(tr_loader.X) > 0 else None
        
        print(f"Train samples: {len(tr[0])}, shape: {tr_loader.X.shape}")
        print(f"Test30 samples: {len(te30[0])}, shape: {te30_loader.X.shape}")
        print(f"Feature dimension: {dim}")
        
        return tr_loader, te30_loader, dim
    else:
        te30_loader = _L(te30)
        dim = te30_loader.X.shape[2] if len(te30[0]) > 0 else None
        
        print(f"Test30 samples: {len(te30[0])}, shape: {te30_loader.X.shape}")
        print(f"Feature dimension: {dim}")
        
        return te30_loader, dim


In [8]:
import time, json, traceback, gc, itertools
from typing import List, Tuple, Optional, Dict
from tqdm import tqdm
from jax.nn.initializers import lecun_normal, normal
from jax.scipy.linalg import block_diag

class GLU(eqx.Module):
    w1: eqx.nn.Linear
    w2: eqx.nn.Linear
    def __init__(self, input_dim, output_dim, key):
        w1_key, w2_key = jr.split(key, 2)
        self.w1 = eqx.nn.Linear(input_dim, output_dim, use_bias=True, key=w1_key)
        self.w2 = eqx.nn.Linear(input_dim, output_dim, use_bias=True, key=w2_key)
    def __call__(self, x):
        return self.w1(x) * jax.nn.sigmoid(self.w2(x))
        
def make_HiPPO(N):
    """Create a HiPPO-LegS matrix.
    From https://github.com/srush/annotated-s4/blob/main/s4/s4.py
    Args:
        N (int32): state size
    Returns:
        N x N HiPPO LegS matrix
    """
    P = jnp.sqrt(1 + 2 * jnp.arange(N))
    A = P[:, jnp.newaxis] * P[jnp.newaxis, :]
    A = jnp.tril(A) - jnp.diag(jnp.arange(N))
    return -A


def make_NPLR_HiPPO(N):
    """
    Makes components needed for NPLR representation of HiPPO-LegS
     From https://github.com/srush/annotated-s4/blob/main/s4/s4.py
    Args:
        N (int32): state size

    Returns:
        N x N HiPPO LegS matrix, low-rank factor P, HiPPO input matrix B

    """
    # Make -HiPPO
    hippo = make_HiPPO(N)

    # Add in a rank 1 term. Makes it Normal.
    P = jnp.sqrt(jnp.arange(N) + 0.5)

    # HiPPO also specifies the B matrix
    B = jnp.sqrt(2 * jnp.arange(N) + 1.0)
    return hippo, P, B


def make_DPLR_HiPPO(N):
    """
    Makes components needed for DPLR representation of HiPPO-LegS
     From https://github.com/srush/annotated-s4/blob/main/s4/s4.py
    Note, we will only use the diagonal part
    Args:
        N:

    Returns:
        eigenvalues Lambda, low-rank term P, conjugated HiPPO input matrix B,
        eigenvectors V, HiPPO B pre-conjugation

    """
    A, P, B = make_NPLR_HiPPO(N)
    S = A + P[:, jnp.newaxis] * P[jnp.newaxis, :]

    S_diag = jnp.diagonal(S)
    Lambda_real = jnp.mean(S_diag) * jnp.ones_like(S_diag)

    # Diagonalize S to V \Lambda V^*
    Lambda_imag, V = jnp.linalg.eigh(S * -1j)

    P = V.conj().T @ P
    B_orig = B
    B = V.conj().T @ B
    return Lambda_real + 1j * Lambda_imag, P, B, V, B_orig


def log_step_initializer(dt_min=0.001, dt_max=0.1):
    """Initialize the learnable timescale Delta by sampling
    uniformly between dt_min and dt_max.
    Args:
        dt_min (float32): minimum value
        dt_max (float32): maximum value
    Returns:
        init function
    """

    def init(key, shape):
        """Init function
        Args:
            key: jax jr key
            shape tuple: desired shape
        Returns:
            sampled log_step (float32)
        """
        return jr.uniform(key, shape) * (jnp.log(dt_max) - jnp.log(dt_min)) + jnp.log(
            dt_min
        )

    return init


def init_log_steps(key, input):
    """Initialize an array of learnable timescale parameters
    Args:
        key: jax jr key
        input: tuple containing the array shape H and
               dt_min and dt_max
    Returns:
        initialized array of timescales (float32): (H,)
    """
    H, dt_min, dt_max = input
    log_steps = []
    for i in range(H):
        key, skey = jr.split(key)
        log_step = log_step_initializer(dt_min=dt_min, dt_max=dt_max)(skey, shape=(1,))
        log_steps.append(log_step)

    return jnp.array(log_steps)


def init_VinvB(init_fun, rng, shape, Vinv):
    """Initialize B_tilde=V^{-1}B. First samples B. Then compute V^{-1}B.
    Note we will parameterize this with two different matrices for complex
    numbers.
     Args:
         init_fun:  the initialization function to use, e.g. lecun_normal()
         rng:       jax jr key to be used with init function.
         shape (tuple): desired shape  (P,H)
         Vinv: (complex64)     the inverse eigenvectors used for initialization
     Returns:
         B_tilde (complex64) of shape (P,H,2)
    """
    B = init_fun(rng, shape)
    VinvB = Vinv @ B
    VinvB_real = VinvB.real
    VinvB_imag = VinvB.imag
    return jnp.concatenate((VinvB_real[..., None], VinvB_imag[..., None]), axis=-1)


def trunc_standard_normal(key, shape):
    """Sample C with a truncated normal distribution with standard deviation 1.
    Args:
        key: jax jr key
        shape (tuple): desired shape, of length 3, (H,P,_)
    Returns:
        sampled C matrix (float32) of shape (H,P,2) (for complex parameterization)
    """
    H, P, _ = shape
    Cs = []
    for i in range(H):
        key, skey = jr.split(key)
        C = lecun_normal()(skey, shape=(1, P, 2))
        Cs.append(C)
    return jnp.array(Cs)[:, 0]


def init_CV(init_fun, rng, shape, V):
    """Initialize C_tilde=CV. First sample C. Then compute CV.
    Note we will parameterize this with two different matrices for complex
    numbers.
     Args:
         init_fun:  the initialization function to use, e.g. lecun_normal()
         rng:       jax jr key to be used with init function.
         shape (tuple): desired shape  (H,P)
         V: (complex64)     the eigenvectors used for initialization
     Returns:
         C_tilde (complex64) of shape (H,P,2)
    """
    C_ = init_fun(rng, shape)
    C = C_[..., 0] + 1j * C_[..., 1]
    CV = C @ V
    CV_real = CV.real
    CV_imag = CV.imag
    return jnp.concatenate((CV_real[..., None], CV_imag[..., None]), axis=-1)


# Discretization functions
def discretize_bilinear(Lambda, B_tilde, Delta):
    """Discretize a diagonalized, continuous-time linear SSM
    using bilinear transform method.
    Args:
        Lambda (complex64): diagonal state matrix              (P,)
        B_tilde (complex64): input matrix                      (P, H)
        Delta (float32): discretisation step sizes             (P,)
    Returns:
        discretized Lambda_bar (complex64), B_bar (complex64)  (P,), (P,H)
    """
    Identity = jnp.ones(Lambda.shape[0])

    BL = 1 / (Identity - (Delta / 2.0) * Lambda)
    Lambda_bar = BL * (Identity + (Delta / 2.0) * Lambda)
    B_bar = (BL * Delta)[..., None] * B_tilde
    return Lambda_bar, B_bar


def discretize_zoh(Lambda, B_tilde, Delta):
    """Discretize a diagonalized, continuous-time linear SSM
    using zero-order hold method.
    Args:
        Lambda (complex64): diagonal state matrix              (P,)
        B_tilde (complex64): input matrix                      (P, H)
        Delta (float32): discretisation step sizes             (P,)
    Returns:
        discretized Lambda_bar (complex64), B_bar (complex64)  (P,), (P,H)
    """
    Identity = jnp.ones(Lambda.shape[0])
    Lambda_bar = jnp.exp(Lambda * Delta)
    B_bar = (1 / Lambda * (Lambda_bar - Identity))[..., None] * B_tilde
    return Lambda_bar, B_bar


# Parallel scan operations
@jax.vmap
def binary_operator(q_i, q_j):
    """Binary operator for parallel scan of linear recurrence. Assumes a diagonal matrix A.
    Args:
        q_i: tuple containing A_i and Bu_i at position i       (P,), (P,)
        q_j: tuple containing A_j and Bu_j at position j       (P,), (P,)
    Returns:
        new element ( A_out, Bu_out )
    """
    A_i, b_i = q_i
    A_j, b_j = q_j
    return A_j * A_i, A_j * b_i + b_j


def apply_ssm(Lambda_bar, B_bar, C_tilde, input_sequence, conj_sym):
    """Compute the LxH output of discretized SSM given an LxH input.
    Args:
        Lambda_bar (complex64): discretized diagonal state matrix    (P,)
        B_bar      (complex64): discretized input matrix             (P, H)
        C_tilde    (complex64): output matrix                        (H, P)
        input_sequence (float32): input sequence of features         (L, H)
        conj_sym (bool):         whether conjugate symmetry is enforced
    Returns:
        ys (float32): the SSM outputs (S5 layer preactivations)      (L, H)
    """
    Lambda_elements = Lambda_bar * jnp.ones(
        (input_sequence.shape[0], Lambda_bar.shape[0])
    )
    Bu_elements = jax.vmap(lambda u: B_bar @ u)(input_sequence)

    _, xs = jax.lax.associative_scan(binary_operator, (Lambda_elements, Bu_elements))

    if conj_sym:
        return jax.vmap(lambda x: 2 * (C_tilde @ x).real)(xs)
    else:
        return jax.vmap(lambda x: (C_tilde @ x).real)(xs)


class S5Layer(eqx.Module):
    Lambda_re: jax.Array
    Lambda_im: jax.Array
    B: jax.Array
    C: jax.Array
    D: jax.Array
    log_step: jax.Array

    H: int
    P: int
    conj_sym: bool
    clip_eigs: bool = False
    discretisation: str
    step_rescale: float = 1.0

    def __init__(
        self,
        ssm_size,
        blocks,
        H,
        C_init,
        conj_sym,
        clip_eigs,
        discretisation,
        dt_min,
        dt_max,
        step_rescale,
        *,
        key
    ):

        B_key, C_key, D_key, step_key, key = jr.split(key, 5)

        block_size = int(ssm_size / blocks)
        # Initialize state matrix A using approximation to HiPPO-LegS matrix
        Lambda, _, B, V, B_orig = make_DPLR_HiPPO(block_size)

        if conj_sym:
            block_size = block_size // 2
            P = ssm_size // 2
        else:
            P = ssm_size

        Lambda = Lambda[:block_size]
        V = V[:, :block_size]
        Vc = V.conj().T

        Lambda = (Lambda * jnp.ones((blocks, block_size))).ravel()
        V = block_diag(*([V] * blocks))
        Vinv = block_diag(*([Vc] * blocks))

        self.H = H
        self.P = P
        if conj_sym:
            local_P = 2 * P
        else:
            local_P = P

        self.Lambda_re = Lambda.real
        self.Lambda_im = Lambda.imag

        self.conj_sym = conj_sym

        self.clip_eigs = clip_eigs

        self.B = init_VinvB(lecun_normal(), B_key, (local_P, self.H), Vinv)

        # Initialize state to output (C) matrix
        if C_init in ["trunc_standard_normal"]:
            C_init = trunc_standard_normal
        elif C_init in ["lecun_normal"]:
            C_init = lecun_normal()
        elif C_init in ["complex_normal"]:
            C_init = normal(stddev=0.5**0.5)
        else:
            raise NotImplementedError("C_init method {} not implemented".format(C_init))

        if C_init in ["complex_normal"]:
            self.C = C_init(C_key, (self.H, 2 * self.P, 2))
        else:
            self.C = init_CV(C_init, C_key, (self.H, local_P, 2), V)

        self.D = normal(stddev=1.0)(D_key, (self.H,))

        # Initialize learnable discretisation timescale value
        self.log_step = init_log_steps(step_key, (self.P, dt_min, dt_max))

        self.step_rescale = step_rescale
        self.discretisation = discretisation

    def __call__(self, input_sequence):
        if self.clip_eigs:
            Lambda = jnp.clip(self.Lambda_re, None, -1e-4) + 1j * self.Lambda_im
        else:
            Lambda = self.Lambda_re + 1j * self.Lambda_im

        B_tilde = self.B[..., 0] + 1j * self.B[..., 1]
        C_tilde = self.C[..., 0] + 1j * self.C[..., 1]

        step = self.step_rescale * jnp.exp(self.log_step[:, 0])

        # Discretize
        if self.discretisation in ["zoh"]:
            Lambda_bar, B_bar = discretize_zoh(Lambda, B_tilde, step)
        elif self.discretisation in ["bilinear"]:
            Lambda_bar, B_bar = discretize_bilinear(Lambda, B_tilde, step)
        else:
            raise NotImplementedError(
                "Discretization method {} not implemented".format(self.discretisation)
            )

        ys = apply_ssm(Lambda_bar, B_bar, C_tilde, input_sequence, self.conj_sym)

        # Add feedthrough matrix output Du;
        Du = jax.vmap(lambda u: self.D * u)(input_sequence)
        return ys + Du

class S5Block(eqx.Module):
    norm: eqx.nn.BatchNorm
    ssm: S5Layer
    glu: GLU
    drop: eqx.nn.Dropout

    def __init__(
        self,
        ssm_size,
        blocks,
        H,
        C_init,
        conj_sym,
        clip_eigs,
        discretisation,
        dt_min,
        dt_max,
        step_rescale,
        drop_rate=0.1,
        *,
        key
    ):
        ssmkey, glukey = jr.split(key, 2)
        

        self.norm = eqx.nn.BatchNorm(
            input_size=H,  axis_name="batch", channelwise_affine=False
        )
        self.ssm = S5Layer(
            ssm_size,
            blocks,
            H,
            C_init,
            conj_sym,
            clip_eigs,
            discretisation,
            dt_min,
            dt_max,
            step_rescale,
            key=ssmkey,
        )
        self.glu = GLU(H, H, key=glukey)
        self.drop = eqx.nn.Dropout(p=drop_rate)

    def __call__(self, x, state, *, key):
        """Compute S5 block."""
        dropkey1, dropkey2 = jr.split(key, 2)
        skip = x
        x, state = self.norm(x.T, state)
        x = x.T
        x = self.ssm(x)
        x = self.drop(jax.nn.gelu(x), key=dropkey1)
        x = jax.vmap(self.glu)(x)
        x = self.drop(x, key=dropkey2)
        x = skip + x
        return x, state


class S5(eqx.Module):
    linear_encoder: eqx.nn.Linear
    blocks: List[S5Block]
    linear_layer_classification: eqx.nn.Linear
    linear_layer_regression: eqx.nn.Linear
    
    classification: bool
    output_step: int
    lambd: float
    lip2: bool = True
    
    def __init__(
        self,
        num_blocks,
        N,
        ssm_size,
        ssm_blocks,
        H,
        output_dim_classification,
        output_dim_regression,
        classification,
        output_step,
        C_init,
        conj_sym,
        clip_eigs,
        discretisation,
        dt_min,
        dt_max,
        step_rescale,
        lambd,
        drop_rate,  
        key
    ):
        linear_encoder_key, *block_keys, linear_layer_classification_key, linear_layer_regression_key = jr.split(
            key, num_blocks + 3
        )
       
        self.linear_encoder = eqx.nn.Linear(N, H, key=linear_encoder_key)
        
        self.blocks = [
            S5Block(
                ssm_size,
                ssm_blocks,
                H,
                C_init,
                conj_sym,
                clip_eigs,
                discretisation,
                dt_min,
                dt_max,
                step_rescale,
                drop_rate=drop_rate,  
                key=key,
            )
            for key in block_keys
        ]
        self.linear_layer_classification = eqx.nn.Linear(H, output_dim_classification, key=linear_layer_classification_key)
        self.linear_layer_regression = eqx.nn.Linear(H, output_dim_regression, key=linear_layer_regression_key)
        self.classification = classification
        self.output_step = output_step
        self.lambd = lambd


    def __call__(self, x, state, key):
        """Compute S5."""
        dropkeys = jr.split(key, len(self.blocks))
        x = jax.vmap(self.linear_encoder)(x)

        for block, key in zip(self.blocks, dropkeys):
            x, state = block(x, state, key=key)
            
        if self.classification:
            x = jnp.mean(x, axis=0)       # mean
            x = self.linear_layer_classification(x)
        else:
            x = jnp.mean(x, axis=0)  
            # x = x[-1]   # mean/sum
            xy = self.linear_layer_regression(x)  # (2,) 单位圆         
            
            norm = jnp.sqrt(jnp.sum(xy**2)) + 1e-8  # 防止除零
            cos_theta = xy[0] / norm
            sin_theta = xy[1] / norm
            x = jnp.array([cos_theta, sin_theta])

        return x, state
        

In [9]:

@eqx.filter_jit
def train_step(model, state, x, y, opt, opt_state, subkey): 
    def loss_fn(m, s, subkey):
        keys = jr.split(subkey, x.shape[0])
        preds, new_state = jax.vmap(m, axis_name="batch",
                                    in_axes=(0, None, 0), 
                                    out_axes=(0, None))(x, s, keys)
        loss = optax.softmax_cross_entropy_with_integer_labels(preds, y).mean()
        params = eqx.filter(m, eqx.is_inexact_array)
        l2_loss = sum(jnp.sum(jnp.square(p)) for p in jax.tree_util.tree_leaves(params))
        loss = loss + m.lambd * l2_loss
        acc = jnp.mean(jnp.argmax(preds, axis=-1) == y)
        return loss, (new_state, acc)
    (loss, (state, acc)), grads = eqx.filter_value_and_grad(loss_fn, has_aux=True)(model, state, subkey)
    updates, opt_state = opt.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, state, opt_state, loss, acc
    
def _get_logits_and_acc(model, state, loader, batch_size=512, key_seed=123, shuffle=False):
    logits_list, targets_list = [], []
    eval_key = jr.PRNGKey(key_seed)
    for x, y, _ in loader.loop(batch_size, eval_key, shuffle=shuffle):
        eval_key, sk = jr.split(eval_key)
        keys = jr.split(sk, x.shape[0])
        p, _ = jax.vmap(model,
                        axis_name="batch",
                        in_axes=(0, None, 0),
                        out_axes=(0, None))(x, state, keys)
        logits_list.append(np.array(p))
        targets_list.append(np.array(y))
    all_logits = np.concatenate(logits_list, axis=0)
    all_targets = np.concatenate(targets_list, axis=0)
    acc = np.mean(np.argmax(all_logits, axis=-1) == all_targets)
    return all_logits, all_targets, float(acc)


def train_epoch(model, state, loader, opt, opt_state, key):
    losses, accs, steps = [], [], 0
    for x, y, _ in loader.loop(loader.batch_size, key, shuffle=True):
        key, subkey = jr.split(key)
        model, state, opt_state, loss, acc = train_step(model, state, x, y, opt, opt_state, subkey)
        losses.append(float(loss))
        accs.append(float(acc))
        steps += 1
    return model, state, opt_state, np.mean(losses), np.mean(accs), steps

def train_loop(model, state, loader, epochs, lr_fn, save_dir, ckpt_every, print_every, key):
    opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(lr_fn))
    opt_state = opt.init(eqx.filter(model, eqx.is_inexact_array))
    log, gpu_mem_records = {"epoch": [], "loss": [], "acc": [], "time": [], "mem": []}, []
    total_steps, t0_total = 0, time.time()

    for ep in range(epochs):
        t0_epoch = time.time()
        key, epoch_key = jr.split(key)
        model, state, opt_state, l, a, steps_in_epoch = train_epoch(
            model, state, loader, opt, opt_state, epoch_key
        )

        total_steps += steps_in_epoch
        cur_mem = get_gpu_memory_usage_mb()
        epoch_time = time.time() - t0_epoch
        gpu_mem_records.append(cur_mem)

        if (ep + 1) % print_every == 0 or ep == 0:
            print(f"Ep {ep+1}/{epochs} | Loss: {l:.4f} | Acc: {a:.2%} | Mem: {cur_mem:.2f}MB | Time：{epoch_time :.2f}s")

        log["epoch"].append(ep + 1)
        log["loss"].append(float(l))
        log["acc"].append(float(a))
        log["time"].append(epoch_time)
        log["mem"].append(cur_mem)

        if save_dir and ckpt_every and (ep + 1) % ckpt_every == 0:
            p = os.path.join(save_dir, "checkpoints")
            os.makedirs(p, exist_ok=True)
            eqx.tree_serialise_leaves(os.path.join(p, f"ep{ep+1}.eqx"), model)
            eqx.tree_serialise_leaves(os.path.join(p, f"state_ep{ep+1}.eqx"), state)
            eqx.tree_serialise_leaves(os.path.join(p, f"opt_state_ep{ep+1}.eqx"), opt_state)

    total_time = time.time() - t0_total
    avg_mem = float(np.mean(log["mem"])) if gpu_mem_records else 0.0
    time_per_1000_steps = (total_time / total_steps * 1000) if total_steps > 0 else 0.0

    extra_stats = {
        "total_time": total_time,
        "total_steps": total_steps,
        "avg_mem": avg_mem,
        "time_per_1000_steps": time_per_1000_steps
    }

    print("\n=== Training Stats ===")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Total Steps: {total_steps}")
    print(f"Time per 1000 steps: {time_per_1000_steps:.2f}s")
    print(f"Avg GPU Memory: {avg_mem:.2f}MB")

    return model, state, log, extra_stats



In [10]:

def compute_ensemble_metrics(results_list):
    if not results_list: return {}
    accs = [r['acc'] for r in results_list]
    
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    
    all_logits = np.stack([r['logits'] for r in results_list], axis=0) # [S, N, C]
    targets = results_list[0]['targets']
    
    # Soft Voting
    probs = np.exp(all_logits) / np.sum(np.exp(all_logits), axis=-1, keepdims=True)
    avg_probs = np.mean(probs, axis=0)
    soft_acc = np.mean(np.argmax(avg_probs, -1) == targets)
    
    # Hard Voting
    seed_preds = np.argmax(all_logits, axis=-1) # [S, N]
    hard_preds = []
    for i in range(seed_preds.shape[1]):
        counts = np.bincount(seed_preds[:, i])
        hard_preds.append(np.argmax(counts))
    hard_acc = np.mean(np.array(hard_preds) == targets)
    
    return {"mean": mean_acc, "std": std_acc, "soft": soft_acc, "hard": hard_acc}
    
def run_single_model(tr_load, te30_load, data_dim, n_cls, save_dir, cfg, seed):
    if os.path.exists(os.path.join(save_dir, "_COMPLETED")):
        print(f"Skipping {save_dir} (already completed)")
        return None

    os.makedirs(save_dir, exist_ok=True)
    print(f"Data dimension: {data_dim}")

    key = jr.PRNGKey(seed)
    modelkey, trainkey = jr.split(key)

    model = S5(
        num_blocks      = cfg['num_blocks'],
        N               = data_dim,
        ssm_size        = cfg['ssm_size'],
        ssm_blocks      = cfg['ssm_blocks'],
        H               = cfg['hidden_dim'],
        output_dim_classification = n_cls,
        output_dim_regression = 2,
        classification  = True,
        output_step     = 1,

        C_init          = "lecun_normal",
        conj_sym        = True,
        clip_eigs       = True,
        discretisation  = "zoh",
        dt_min          = 0.001,
        dt_max          = 0.1,
        step_rescale    = 1.0,
        lambd           = cfg['lambd'],
        drop_rate       = cfg['drop_rate'],
        key             = modelkey
    )


    state = eqx.nn.State(model)
    lr = optax.exponential_decay(1e-3, 4500, 0.5, staircase=True, end_value=1e-6)

    model, state, log, stats = train_loop(
        model, state,
        tr_load,
        cfg['total_epochs'],
        lr,
        save_dir,
        cfg['save_ckpt_every'],
        cfg['print_every'],
        key=trainkey
    )

    print("\n=== Evaluating on 30° Rotated Test Set ===")
    l30, t30, a30 = _get_logits_and_acc(model, state, te30_load)
    print(f"30° Test Accuracy: {a30:.4f}")

    eqx.tree_serialise_leaves(os.path.join(save_dir, "model.eqx"), model)
    eqx.tree_serialise_leaves(os.path.join(save_dir, "state.eqx"), state)

    res_summary = {
        "cfg": cfg,
        "seed": seed,
        "acc30": a30,
        "train_time": stats['total_time'],
        "total_steps": stats['total_steps'],
        "time_per_1000_steps": stats['time_per_1000_steps'],
        "avg_mem": stats['avg_mem']
    }

    with open(os.path.join(save_dir, "summary.json"), "w") as f:
        json.dump(res_summary, f, indent=2)

    open(os.path.join(save_dir, "_COMPLETED"), "w").close()

    return {
        "summary": res_summary,
        "res_30x": {"logits": l30, "targets": t30, "acc": a30}
    }



def run_experiment(data_dir, n_cls, save_dir, cfg):
    seeds = cfg.pop('seed_list')

    print(f">>> Single Run. Seeds: {seeds}")
    print("Loading and splitting data (once for all seeds)...")
    
    tr_load, te30_load, dim  = make_loaders(data_dir, n_cls, cfg, seed=42, mode="train")
    data_dim = dim[-1] if isinstance(dim, tuple) else dim
    results = []

    for s in seeds:
        res = run_single_model(tr_load, te30_load, data_dim, n_cls, os.path.join(save_dir, f"seed_{s}"), cfg.copy(), s)
        if res: results.append(res)

    if results:
        m30 = compute_ensemble_metrics([r['res_30x'] for r in results])
        avg_train_time = np.mean([r['summary']['train_time'] for r in results])
        avg_time_per_1000 = np.mean([r['summary']['time_per_1000_steps'] for r in results])
        avg_mem = np.mean([r['summary']['avg_mem'] for r in results])
        print("\n=== FINAL RESULTS ===")
        print(f"Test 30x: Mean±Std {m30['mean']:.4f} ± {m30['std']:.4f}")
        print(f"Soft Voting: {m30['soft']:.4f}")
        print(f"Hard Voting: {m30['hard']:.4f}")

def grid_search(data_dir, n_cls, base_dir, base_cfg, grid_vars):
    keys, values = zip(*grid_vars.items())
    all_combos = itertools.product(*values)
    combos = [
        dict(zip(keys, v))
        for v in all_combos
        if (v[keys.index("ssm_size")] % v[keys.index("ssm_blocks")] == 0)
    ]
    print(f">>> Starting Grid Search: {len(combos)} valid combinations (filtered)")
    master_csv = os.path.join(base_dir, "grid_summary.csv")

    for i, c in enumerate(combos):
        cfg = base_cfg.copy()
        cfg.update(c)
        prefix = f"Exp{i}_" + "_".join([f"{k}{v}" for k, v in c.items() if k not in ['seed_list']])
        exp_path = os.path.join(base_dir, prefix)

        print(f"\n=== [{i+1}/{len(combos)}] Running {prefix} ===")

        tr_load, te30_load, dim = make_loaders(data_dir, n_cls, cfg, seed=42, mode="train")
        data_dim = dim[-1] if isinstance(dim, tuple) else dim
        seeds = cfg.pop('seed_list')

        results = []
        for s in seeds:
            save_s = os.path.join(exp_path, f"seed_{s}")
            if os.path.exists(os.path.join(save_s, "_COMPLETED")):
                print(f"  Seed {s} already completed, skipping.")
                continue
            try:
                res = run_single_model(tr_load, te30_load, data_dim, n_cls, save_s, cfg.copy(), s)
                if res:
                    results.append(res)
            except Exception as e:
                print(f"  [ERROR] Skipping combination {prefix}, seed={s}, error={e}")
                continue

        if results:
            m30 = compute_ensemble_metrics([r['res_30x'] for r in results])
            avg_time = np.mean([r['summary']['train_time'] for r in results])
            avg_time_per_1000 = np.mean([r['summary']['time_per_1000_steps'] for r in results])
            avg_mem = np.mean([r['summary']['avg_mem'] for r in results])

            row = {
                **c,
                "acc30_mean": m30['mean'],
                "acc30_std": m30['std'],
                "acc30_soft": m30['soft'],
                "acc30_hard": m30['hard'],
                "avg_train_time": avg_time,
                "avg_time_per_1000_steps": avg_time_per_1000,
                "avg_gpu_mem": avg_mem,
            }

            pd.DataFrame([row]).to_csv(
                master_csv,
                mode='a',
                header=not os.path.exists(master_csv),
                index=False
            )

    print("\nGrid search completed.")




In [11]:
if __name__ == "__main__":
    DATA_DIR = "RotFreeData/EngUpper26" # EngUpper26 / Digits10 / Radicals52
    SAVE_DIR = "Model/S5/"
    NUM_CLASSES = 26

    BASE_CONFIG = {
        # Preprocessing
        "use_simplify": True, "dist_thresh": 5.0, "angle_thresh": 0.15,
        "use_speed_norm": True, "target_speed": 10.0,
        "coord_range": 1.0,
        "add_ink_dim": True, "add_pen_states": True, "add_imaginary_stroke": False,
        "use_derivatives": True, "use_end_padding": True, "use_signatures": True,

        # Training
        "print_every": 20, "save_ckpt_every": 0,
        
        # Model
        "num_blocks": 6, "ssm_size": 512, "ssm_blocks": 4,"hidden_dim": 512,
         "drop_rate": 0.3, "lambd": 1e-4,
    
    }


    VARS = {
        "target_length": [100], 
        "window_size":   [5],
        "stride":        [1],
        "depth":         [2],
        
        "use_hanging":   [True],
        "hanging_mode":  ["SC"],
    
        "total_epochs":  [200],
        "batch_size":    [64],
    
        "seed_list":     [[1011, 1012, 1013, 1014, 1015]] 
    }

    MODE = "SINGLE"  # "GRID" or "SINGLE"
    if MODE == "SINGLE":
        cfg = BASE_CONFIG.copy()
        for k, v in VARS.items():
            cfg[k] = v[0]
        run_experiment(DATA_DIR, NUM_CLASSES, SAVE_DIR, cfg)
    
    elif MODE == "GRID":
        grid_search(DATA_DIR, NUM_CLASSES, SAVE_DIR, BASE_CONFIG, VARS)


>>> Single Run. Seeds: [1011, 1012, 1013, 1014, 1015]
Loading and splitting data (once for all seeds)...


2025-12-16 07:24:17.139360: W external/xla/xla/service/gpu/nvptx_compiler.cc:765] The NVIDIA driver's CUDA version is 12.2 which is older than the ptxas CUDA version (12.8.93). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
Build train set: 100%|██████████████████████████████████████████████████████████████████| 26/26 [01:10<00:00,  2.73s/it]


Train samples: 6237, shape: (6237, 96, 90)
Test30 samples: 46800, shape: (46800, 96, 90)
Feature dimension: 90
Skipping Model/S5/seed_1011 (already completed)
Skipping Model/S5/seed_1012 (already completed)
Skipping Model/S5/seed_1013 (already completed)
Skipping Model/S5/seed_1014 (already completed)
Data dimension: 90


/home/zheling/anaconda3/envs/py39/lib/python3.9/site-packages/jax/_src/lax/lax.py:2785: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)
/home/zheling/anaconda3/envs/py39/lib/python3.9/site-packages/jax/_src/lax/lax.py:2785: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


Ep 1/200 | Loss: 79694.1006 | Acc: 55.06% | Mem: 33331.00MB | Time：48.41s
Ep 20/200 | Loss: 79562.8746 | Acc: 97.40% | Mem: 33331.00MB | Time：2.42s
Ep 40/200 | Loss: 79430.6103 | Acc: 98.69% | Mem: 33331.00MB | Time：2.37s
Ep 60/200 | Loss: 79344.7202 | Acc: 99.65% | Mem: 33331.00MB | Time：2.41s
Ep 80/200 | Loss: 79279.9810 | Acc: 99.62% | Mem: 33331.00MB | Time：2.30s
Ep 100/200 | Loss: 79223.4507 | Acc: 99.86% | Mem: 33331.00MB | Time：2.40s
Ep 120/200 | Loss: 79179.5499 | Acc: 99.86% | Mem: 33331.00MB | Time：2.43s
Ep 140/200 | Loss: 79138.6445 | Acc: 99.98% | Mem: 33331.00MB | Time：2.42s
Ep 160/200 | Loss: 79128.7202 | Acc: 99.81% | Mem: 33331.00MB | Time：2.41s
Ep 180/200 | Loss: 79118.7979 | Acc: 99.94% | Mem: 33331.00MB | Time：2.41s
Ep 200/200 | Loss: 79111.5677 | Acc: 99.97% | Mem: 33331.00MB | Time：2.37s

=== Training Stats ===
Total Time: 531.68s
Total Steps: 19600
Time per 1000 steps: 27.13s
Avg GPU Memory: 33331.00MB

=== Evaluating on 30° Rotated Test Set ===
30° Test Accuracy: